# RAG A/B Tester — Debug Notebook
Step-by-step tests for WatsonX AI and OpenSearch connectivity, ingestion, and querying.

In [ ]:
import sys, os
from pathlib import Path

# Make sure project root is on the path
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

## 1 · Environment & Credentials

In [ ]:
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

print("WATSONX_URL          :", os.getenv("WATSONX_URL"))
print("WATSONX_LLM_MODEL_ID :", os.getenv("WATSONX_LLM_MODEL_ID"))
print("WATSONX_EMBED_MODEL_ID:", os.getenv("WATSONX_EMBED_MODEL_ID"))
print("OPENSEARCH_URL       :", os.getenv("OPENSEARCH_URL"))
print("OPENSEARCH_USERNAME  :", os.getenv("OPENSEARCH_USERNAME"))
print("OPENSEARCH_PASSWORD  :", "***" if os.getenv("OPENSEARCH_PASSWORD") else "NOT SET")
print("OPENSEARCH_CA_CERT   :", os.getenv("OPENSEARCH_CA_CERT"))
print("OPENSEARCH_VERIFY_SSL:", os.getenv("OPENSEARCH_VERIFY_SSL"))

## 2 · Parse & validate the OpenSearch URL

In [ ]:
from urllib.parse import urlparse

raw = os.getenv("OPENSEARCH_URL", "").strip()
if not raw.startswith(("http://", "https://")):
    raw = "https://" + raw

p = urlparse(raw)
print(f"scheme   : {p.scheme}")
print(f"hostname : {p.hostname}")
print(f"port     : {p.port}")
print(f"path     : '{p.path}'   ← should be empty or '/'")
print()
if p.path and p.path not in ("/", ""):
    print("⚠️  URL has a path component — this will cause 404 errors.")
    print("   OpenSearch API calls go to <url>/<index>/_doc etc.")
    print("   Set OPENSEARCH_URL to just the host:port with no path.")
else:
    print("✅ URL looks well-formed.")

## 3 · Direct ZenApiKey connection test (no config layer)

In [ ]:
from opensearchpy import OpenSearch

# ── Paste your values directly here to test without .env ──────────────────
OPENSEARCH_URL  = os.getenv("OPENSEARCH_URL", "").strip()
OPENSEARCH_KEY  = os.getenv("OPENSEARCH_API_KEY", "")   # IBM Cloud IAM API key

print(f"URL : {OPENSEARCH_URL}")
print(f"Key : {'SET (' + str(len(OPENSEARCH_KEY)) + ' chars)' if OPENSEARCH_KEY else 'NOT SET ❌'}")
print()

client = OpenSearch(
    hosts=[OPENSEARCH_URL],
    headers={"Authorization": f"ZenApiKey {OPENSEARCH_KEY}"},
    use_ssl=True,
    verify_certs=True,       # Let's Encrypt cert — trusted by default
    ssl_show_warn=False,
)

try:
    info = client.info()
    print("✅ Connected!")
    print(f"   Cluster : {info['cluster_name']}")
    print(f"   Version : {info['version']['number']}")
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")
    print()
    print("Troubleshooting:")
    print("  401 → API key is wrong or expired. Generate a new one at:")
    print("        cloud.ibm.com → Manage → Access (IAM) → API keys")
    print("  SSL → Try adding:  verify_certs=False  (temporary test only)")

## 4 · Verify the API key works via raw HTTP (no opensearch-py)

In [ ]:
import urllib.request, json

url = os.getenv("OPENSEARCH_URL", "").strip().rstrip("/")
key = os.getenv("OPENSEARCH_API_KEY", "")

req = urllib.request.Request(
    url,
    headers={"Authorization": f"ZenApiKey {key}"}
)

try:
    with urllib.request.urlopen(req, timeout=10) as resp:
        body = json.loads(resp.read())
        print("✅ HTTP GET / succeeded")
        print(json.dumps(body, indent=2))
except urllib.error.HTTPError as e:
    body = e.read().decode()
    print(f"❌ HTTP {e.code}: {body}")
    if e.code == 401:
        print()
        print("The API key is being rejected by the IBM gateway.")
        print("Check that OPENSEARCH_API_KEY in your .env is a valid IBM Cloud IAM API key")
        print("(not the WatsonX API key, not a username — a standalone IAM API key).")
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

## 5 · List existing indexes

In [ ]:
from src.config import get_opensearch_client

client = get_opensearch_client()
indices = client.cat.indices(format="json")

if indices:
    print(f"{'Index':<50} {'Docs':>8} {'Size':>10}")
    print("-" * 72)
    for idx in sorted(indices, key=lambda x: x.get("index", "")):
        print(f"{idx['index']:<50} {idx.get('docs.count','?'):>8} {idx.get('store.size','?'):>10}")
else:
    print("No indexes found (cluster is empty).")

## 6 · Test WatsonX embeddings

In [ ]:
from src.config import get_embeddings

embed = get_embeddings()
test_text = "What is retrieval-augmented generation?"
vector = embed.embed_query(test_text)

print(f"✅ Embedding dim : {len(vector)}")
print(f"   First 5 vals  : {vector[:5]}")

## 7 · Test WatsonX LLM

In [ ]:
from src.config import get_llm

llm = get_llm(temperature=0)
response = llm.invoke("In one sentence, what is RAG?")
print("✅ LLM response:", response.content)

## 8 · Manually create a test index

In [ ]:
TEST_INDEX = "rag_ab_debug_test"

# Clean up if it exists
if client.indices.exists(index=TEST_INDEX):
    client.indices.delete(index=TEST_INDEX)
    print(f"Deleted existing index: {TEST_INDEX}")

# Create a minimal index
client.indices.create(
    index=TEST_INDEX,
    body={"settings": {"number_of_shards": 1, "number_of_replicas": 0}}
)
print(f"✅ Index created: {TEST_INDEX}")

# Clean up
client.indices.delete(index=TEST_INDEX)
print(f"   Deleted test index.")

## 9 · End-to-end mini ingest (single doc → OpenSearch)

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import OpenSearchVectorSearch
from src.config import get_embeddings, get_opensearch_kwargs

MINI_INDEX = "rag_ab_debug_mini"

docs = [
    Document(page_content="RAG combines retrieval with generation.", metadata={"src": "test"}),
    Document(page_content="Embeddings map text to dense vectors.", metadata={"src": "test"}),
    Document(page_content="OpenSearch stores and searches vectors.", metadata={"src": "test"}),
]

# Clean up first
if client.indices.exists(index=MINI_INDEX):
    client.indices.delete(index=MINI_INDEX)

os_kwargs = get_opensearch_kwargs()
vs = OpenSearchVectorSearch.from_documents(
    docs,
    embedding=get_embeddings(),
    index_name=MINI_INDEX,
    engine="lucene",
    space_type="cosinesimil",
    **os_kwargs,
)
print(f"✅ Indexed {len(docs)} docs into '{MINI_INDEX}'")

## 10 · Query the mini index

In [ ]:
results = vs.similarity_search("How does retrieval work?", k=2)
for i, r in enumerate(results):
    print(f"[{i+1}] {r.page_content}")

# Clean up
client.indices.delete(index=MINI_INDEX)
print("\n✅ Mini index cleaned up.")

## 11 · Run a full pipeline variant (SmallChunkPipeline)

In [ ]:
import importlib.util
from src.config import safe_index

spec = importlib.util.spec_from_file_location("exp", PROJECT_ROOT / "experiments/chunking.py")
exp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp)

pipeline = exp.CONTROL(index_name=safe_index("ctrl", exp.CONTROL_NAME))
pipeline.ingest()
print("✅ Ingested docs into OpenSearch")

answer, chunks = pipeline.query("What is chunking in RAG?")
print("\nAnswer:", answer)
print("\nContext chunks used:")
for i, c in enumerate(chunks):
    print(f"  [{i+1}] {c[:120]}...")